In [3]:
%pip install tensorflow
%pip install scikit-learn
%pip install pytest



[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
import h5py
import os
import hashlib
from pathlib import Path

import tensorflow as tf
import tensorflow.keras as keras
import sklearn as sk
import matplotlib.pyplot as plt
import requests
import pytest
import numpy as np

# Fetching Dataset

The dataset is downloaded from Zenodo at runtime, rather than attached at submission.

- If Galaxy10.h5 exists, its size is checked against the expected size fetched from the request header. If it matches, download is skipped, if not, it is deleted and downloaded again. 
  
- If it doesn't exist, it is streamed in chunks to `Galaxy10.h5.part`, and the hash provided from the docs is compared during the donwload. Once the size and hash is verified, then the machine learning pipeline starts. 
  
- If verification fails, download is retried once, then the program stops with an error.


In [5]:
def stream_ingest(url, fname, hash):
    '''
    to complete
    '''
    try:
        r = requests.get(url, stream=True)
        r.raise_for_status()
    except requests.exceptions.HTTPError as e:
        return ("HTTP error occurred:", e)
    except requests.exceptions.RequestException as e:
        return "A request error occurred:", e
        
    chunk_fname = f'{fname}.part'
    fpath = Path(fname)
    fsize = int(r.headers['content-length'])
    
    hash_func = hashlib.new('sha256')
        
    # ingesting
    if not fpath.is_file() or os.path.getsize(fpath) != fsize:
        with open(chunk_fname, 'wb') as file:
            for chunk in r.iter_content(chunk_size=5_242_880):
                file.write(chunk)
                hash_func.update(chunk)
                print('Downloaded', len(chunk), "bytes")
                
            if hash_func.hexdigest().lower() == hash.lower():
                os.rename(chunk_fname, fname)
            else:
                print(f'Wrong file downloaded due to differing hash, deleting file: {chunk_fname}')
                os.remove(chunk_fname)
            
zenodo_url = 'https://zenodo.org/records/10844811/files/Galaxy10.h5'
HASH = '969A6B1CEFCC36E09FFFA86FEBD2F699A4AA19B837BA0427F01B0BC6DED458AF'.lower()
fname = 'Galaxy10.h5'       
stream_ingest(zenodo_url, fname, HASH)    

## Data Pre-Processing

HDF5 is a file format to store and manage complex data, and is what the dataset is orginally in (see appendix)

Below cells are to manipulate this file format so its readable in Python

In [6]:
def read_to_np(dataset):
    n1 = np.array(dataset)
    print(n1.shape)
    return n1

with h5py.File('Galaxy10.h5', 'r') as hf:
    for group in hf:
        print(hf[group])
        
    X_raw = read_to_np(hf['images'])
    y_raw = read_to_np(hf['ans'])
    

<HDF5 dataset "ans": shape (21785,), type "|u1">
<HDF5 dataset "images": shape (21785, 69, 69, 3), type "|u1">
(21785, 69, 69, 3)
(21785,)


In [7]:
# goal:

#get unique values of y data 
# plot y count of data (histogram)
# try visually display x and research the band 



X (images) has four dimensions. Looking at the shape it is composed of:
```
211785 rows (images)
    inside each: collection of 69 rows (69 pixels across)
        inside each: collection of 69 columns (69 pixels downwards)
            inside each: 3 colour channels (g, r, i band)

```


In [8]:
print('IMAGES STRUCTURE:')
print(f'dimensions- {X_raw.ndim}')
print(X_raw)

print('\nLABEL STRUCTURE:')
print(f'dimensions- {y_raw.ndim}')
print(y_raw)

IMAGES STRUCTURE:
dimensions- 4
[[[[16 18 15]
   [18 20 15]
   [16 17 12]
   ...
   [ 6  6  4]
   [ 9  9  7]
   [ 6  6  4]]

  [[19 24 20]
   [23 25 20]
   [17 18 13]
   ...
   [ 8  5  3]
   [10  8  6]
   [ 6  6  4]]

  [[17 22 18]
   [21 24 19]
   [17 18 13]
   ...
   [ 8  4  3]
   [11  8  7]
   [ 7  6  4]]

  ...

  [[17 14  7]
   [18 17  9]
   [15 14  7]
   ...
   [ 2  4  0]
   [ 3  5  0]
   [ 5  7  2]]

  [[22 14 11]
   [19 14  8]
   [12  9  3]
   ...
   [ 4  7  0]
   [ 6  9  2]
   [ 7 10  3]]

  [[30 12 11]
   [31 14  8]
   [24  9  2]
   ...
   [ 8 11  3]
   [11 14  6]
   [12 15  7]]]


 [[[ 1  3  0]
   [ 1  1  1]
   [ 1  1  1]
   ...
   [ 5  7  4]
   [ 3  5  2]
   [ 4  6  3]]

  [[ 1  2  0]
   [ 1  1  0]
   [ 0  0  0]
   ...
   [ 3  7  3]
   [ 2  6  2]
   [ 2  6  3]]

  [[ 3  3  1]
   [ 1  1  0]
   [ 0  0  0]
   ...
   [ 1  3  0]
   [ 2  4  1]
   [ 2  4  1]]

  ...

  [[ 5  7  4]
   [ 5  7  4]
   [ 3  5  2]
   ...
   [ 2  4  1]
   [ 1  3  1]
   [ 1  3  0]]

  [[ 4  6  3]
   [ 3  

In [9]:
unique, counts = np.unique(y_raw, return_counts=True)
print(np.asarray((unique, counts)).T)

keep = y_raw != 5
# apply to y and 
y_clean = y_raw[keep]
X_clean = X_raw[keep]
print(X_clean.shape, y_clean.shape)

del y_raw, X_raw

[[   0 3461]
 [   1 6997]
 [   2 6292]
 [   3  349]
 [   4 1534]
 [   5   17]
 [   6  589]
 [   7 1121]
 [   8  906]
 [   9  519]]
(21768, 69, 69, 3) (21768,)


Removing Class 5, as it is only 17 of the cases ,and would result in noise/

In [ ]:
print(np.unique(X_clean)) # 0 - 255

X_norm = X_clean.astype(np.float32) / 255 # float32 to save space
print(X_norm)
del X_clean


[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197
 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215
 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233
 234 235 236 237 238 239 240 241 242 243 244 245 24

## Plotting